# RAG Grid Search: 4-Config Experiment

Grid search across 2 axes: **chunk size** (256 vs 512) and **LLM model** (weak vs strong).

Embedding (`all-mpnet-base-v2`), retriever (dense/FAISS), reranker (`CrossEncoderReranker` ms-marco-MiniLM-L6, `top_n=5`), and search (`similarity`, `k=15`) held constant.

| Run | Chunk | Embedding | Retriever | LLM | Purpose |
|-----|-------|-----------|-----------|-----|---------|
| 1 | 256/32 | mpnet-base | dense | `api-mistral-small-3.2-2506` (weak) | baseline |
| 2 | 512/64 | mpnet-base | dense | `api-mistral-small-3.2-2506` (weak) | chunk size effect (vs 1) |
| 3 | 256/32 | mpnet-base | dense | `api-llama-4-scout` (strong) | LLM upgrade (vs 1) |
| 4 | 512/64 | mpnet-base | dense | `api-llama-4-scout` (strong) | dense+strong baseline (vs 2) |

In [1]:
from pathlib import Path
TRITON_API_KEY = Path('~/api-key.txt').expanduser().read_text(encoding='utf-8').splitlines()[0].strip()
BASE_URL = 'https://tritonai-api.ucsd.edu'

In [2]:
from typing import Dict, Any
from typing import List as listtype
import re
from collections import defaultdict
import pandas as pd

In [3]:
# document loaders
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document

# text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# retriever
from langchain_community.retrievers import TFIDFRetriever

# embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# cross encoder for reranking
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

In [4]:
# rapidfire imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

INFO 05-04 20:25:24 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-04 20:25:24 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


In [5]:
from rapidfire_integration_example import sample_compute_metrics_fn, sample_accumulate_metrics_fn
from project1_eval import to_spans

WEAK_MODEL = 'api-mistral-small-3.2-2506'
STRONG_MODEL = 'api-llama-4-scout'

In [6]:
import os

os.environ['OPENAI_API_KEY'] = TRITON_API_KEY
os.environ['JUDGE_BASE_URL'] = BASE_URL
os.environ['JUDGE_MODEL'] = 'api-mistral-small-3.2-2506'

In [7]:
class TextLoaderWithMetadata(TextLoader):
    def __init__(self, file_path: str, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func
    
    def load(self) -> list[Document]:
        # load the documents as TextLoader
        docs = super.load()
        # if a metadata function is specified
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        
        return docs
    
    def lazy_load(self):
        # for each document loaded using TextLoader
        for doc in super().lazy_load():
            # if a metadata function is specified
            if self.metadata_func is not None:
                # apply the metadata function to the document
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc

In [8]:
def metadata_func(document, metadata):
    metadata['source'] = Path(metadata['source']).name
    return metadata

In [9]:
loader = DirectoryLoader(
    path='data/sourcedocs/',
    glob='**/*.rst',
    loader_cls=TextLoaderWithMetadata,
    loader_kwargs={
        'metadata_func': metadata_func,
    },
    show_progress=True,
    use_multithreading=True
)

In [10]:
docs = loader.load()

100%|██████████| 31/31 [00:00<00:00, 1447.94it/s]


In [11]:
class TextSplitterWithLineNum(RecursiveCharacterTextSplitter):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def split_documents(self, documents):
        split = super().split_documents(documents)

        # create a mapping for every document
        line_num_map = {}
        for d in documents:
            file_name = d.metadata['source']
            m = defaultdict(list)
            # iterate through each line and give it a mapping
            for idx, line in enumerate(d.page_content.split('\n')):
                m[line.strip()].append(idx + 1)
            line_num_map[file_name] = m

        # for each chunk find the line index mapping for the first and last line
        for chunk in split:
            source = chunk.metadata['source']
            lines = chunk.page_content.strip().split('\n')
            # get the first and last line of the chunk
            first, last = line_num_map[source][lines[0].strip()], line_num_map[source][lines[-1].strip()]

            # find the start index and 
            start = first.pop(0)
            # choose the first matching end line at or after start
            end = next(i for i in last if i >= start)

            chunk.metadata['source'] = {"file": source, 'lines': [start, end]}
        return split

In [12]:
splitter_256 = TextSplitterWithLineNum.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=256,
    chunk_overlap=32,
    separators=['\n', '\n\n'],
)

splitter_512 = TextSplitterWithLineNum.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=512,
    chunk_overlap=64,
    separators=['\n', '\n\n'],
)

In [13]:
batch_size = 32

mpnet_embed_cfg = {
    'class': HuggingFaceEmbeddings,
    'model_name': 'sentence-transformers/all-mpnet-base-v2',
    'model_kwargs': {'device': 'cpu'},
    'encode_kwargs': {'normalize_embeddings': True, 'batch_size': batch_size},
}

In [14]:
vector_store_cfg={
    'type': 'faiss', 
    'batch_size': batch_size
}

In [15]:
def custom_template(doc):
    return f'{doc.metadata["source"]}: {doc.page_content}'

In [16]:
reranker_cfg = {
    'class': CrossEncoderReranker,
    'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2',
    'model_kwargs': {'device': 'cpu'},
    'top_n': 5,
}
search_cfg = {'type': 'similarity', 'k': 15}

# Config 1: chunk=256, mpnet, dense, WEAK
rag1 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_256,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# Config 2: chunk=512, mpnet, dense, WEAK
rag2 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_512,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# Config 3: chunk=256, mpnet, dense, STRONG
rag3 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_256,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

# Config 4: chunk=512, mpnet, dense, STRONG
rag4 = RFLangChainRagSpec(
    document_loader=loader, text_splitter=splitter_512,
    embedding_cfg=mpnet_embed_cfg, vector_store_cfg=vector_store_cfg,
    search_cfg=search_cfg, reranker_cfg=reranker_cfg,
    enable_gpu_search=False, document_template=custom_template,
)

In [17]:
INSTRUCTIONS = """
You are a helpful assistant answering questions about RapidFire AI docuentation.

Your job:
- Read the retrieved documentation context
- Answer the user's question using only the context
- Do not make up information that is not supported by the retrieved text

Rules:
1. Use only the retrieved document context.
2. Keep the answer concise but complete.
3. The answer should be in plain text and do not use any markdown syntax including list.
4. Mention relevant file names and the corresponding start and end line number when they are clearly useful.
5. There can be multiple sources but only keep the ones that are really relevant.
6. The source must be in valid JSON format.

Output format:
#### ANSWER: <final answer>
#### SOURCE: [{"file": <source file>, "lines": [<start line>, <end line>]}]
"""

In [18]:
import json
from datasets import Dataset

In [19]:
def load_eval_question_set(path, encoding='utf-8', validation=False):
    # load the questions from the file
    with open(path, 'r', encoding=encoding) as f:
        eval_questions = json.load(f)
    
    # to store each example
    eval_rows = []

    for example in eval_questions:
        # extract the question and id
        row = {
            'question_id': example['question_id'],
            'question': example['question']
        }
        # validation sets include reference answer an source evidence
        if validation:
            row['reference_answer'] = example['reference_answer']
            row['source_evidence'] = example['source_evidence']
        
        eval_rows.append(row)

    
    # convert into a hugging face dataset
    eval_dataset = Dataset.from_list(eval_rows)
    return eval_dataset

golden_qa_val = load_eval_question_set(path='data/validation-set/validation-set-golden-qa-pairs.json', validation=True)

In [20]:
def sample_preprocess_fn(batch: Dict[str, listtype], 
                         rag: RFLangChainRagSpec, 
                         prompt_manager: RFPromptManager) -> Dict[str, listtype]:
    '''
    convert a batch of dataset rows into model-ready prompts
    '''
    # retrieve context for each question in the batch
    all_context = rag.get_context(batch_queries=batch['question'], serialize=False)
    # extract the source metadata field and retrieved context from each retrieved document
    retrieved_documents, retrieved_contexts = [], []
    for docs in all_context:
        evidence_for_query, context = [], []
        for doc in docs:
            evidence_for_query.append(doc.metadata.get('source'))
            context.append(doc.page_content)
        # attach the source evidence and retrieved context to the question
        retrieved_documents.append(evidence_for_query)
        retrieved_contexts.append(context)

    # convert retrieved docs into plain text strings suitable for prompt injection
    serialized_context = rag.serialize_documents(all_context)

    # build a chat-style prompt per question
    prompts = [
        [
            # instruction as system prompt
            {'role': 'system', 'content': INSTRUCTIONS},
            # user message containing the question
            {
                'role': 'user', 'content': f'Question: {question} \nDocumentation context: {context} \nYour response:'
            }
        ]
        for question, context in zip(batch['question'], serialized_context)
    ]

    return {
        'prompts': prompts,
        'retrieved_documents': retrieved_documents,
        'retrieved_contexts': retrieved_contexts,
        'serialized_context': serialized_context,
        **batch
    }

In [21]:
def extract_solution(answer: str) -> str:
    '''
    extract the answer
    '''
    # search for the answer marker
    answer_match = re.search(
        r"####\s*Answer:\s*(.*?)(?=\n####\s*Source:|\Z)",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    )
    # search for the source marker
    source_match = re.search(
        r"####\s*Source:\s*(.*)",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    )
    answer = answer_match.group(1).strip() if answer_match else answer
    source = source_match.group(1).strip() if source_match else None
    return answer, source

In [22]:
def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    '''
    turn raw generated text into a field
    '''
    # extract answer and source from the generated text
    answer, source = [], []
    for text in batch['generated_text']:
        a, s = extract_solution(text)
        answer.append(a)
        source.append(s)
    batch['answer'] = answer
    batch['source'] = source
    # just different naming for the metrics
    parsed_sources = []
    for s in batch['source']:
        if s is None:
            parsed_sources.append([])
        else:
            try: 
                parsed_sources.append(json.loads(s))
            except json.JSONDecodeError:
                parsed_sources.append([])
    batch['retrieved_spans'] = [to_spans(s) for s in parsed_sources]
    batch['ground_truth_spans'] = [to_spans(s) for s in batch['source_evidence']]
    batch['query'] = batch['question']
    return batch

In [23]:
client_cfg = {'api_key': TRITON_API_KEY, 'base_url': BASE_URL, 'max_retries': 2}

def make_openai_config(model_name, rag):
    return RFOpenAIAPIModelConfig(
        client_config=client_cfg,
        model_config={'model': model_name, 'max_completion_tokens': 2048, 'reasoning_effort': 'high'},
        rpm_limit=120,
        tpm_limit=1_000_000,
        rag=rag,
        prompt_manager=None,
    )

oc1 = make_openai_config(WEAK_MODEL, rag1)   # chunk=256, weak
oc2 = make_openai_config(WEAK_MODEL, rag2)   # chunk=512, weak
oc3 = make_openai_config(STRONG_MODEL, rag3)  # chunk=256, strong
oc4 = make_openai_config(STRONG_MODEL, rag4)  # chunk=512, strong

CONFIG_LABELS = {
    1: "chunk=256, mpnet, dense, weak",
    2: "chunk=512, mpnet, dense, weak",
    3: "chunk=256, mpnet, dense, strong",
    4: "chunk=512, mpnet, dense, strong",
}

print(f"Built {len(CONFIG_LABELS)} RAG configs")

Built 4 RAG configs


In [24]:
config_set = [{
    'openai_config': List([oc1, oc2, oc3, oc4]),
    'batch_size': batch_size,
    'preprocess_fn': sample_preprocess_fn,
    'postprocess_fn': sample_postprocess_fn,
    'compute_metrics_fn': sample_compute_metrics_fn,
    'accumulate_metrics_fn': sample_accumulate_metrics_fn,
}]

config_group = RFGridSearch(config_set)

In [25]:
# create experiment
experiment = Experiment(
    experiment_name='rag-grid-search-4configs',
    mode='evals'
)

# run the 4-config grid search
results = experiment.run_evals(
    config_group=config_group,
    dataset=golden_qa_val,
    num_shards=4,
    seed=42
)
experiment.end()

The previously running experiment rag-generation-only_12 was forcibly ended. Created a new experiment 'rag-grid-search-4configs_4' with Experiment ID: 23 at /home/emc001/rapidfireai/rapidfire_experiments/rag-grid-search-4configs_4


=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,150.5s,CPU,FAISS
2,Complete,150.6s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.class,embedding_cfg.encode_kwargs.batch_size,embedding_cfg.encode_kwargs.normalize_embeddings,embedding_cfg.model_name,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.k,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Completeness_normalized,Correctness_pass_rate,F1_at_5,Faithfulness_pass_rate,Generation_Score_3_released,Judge Failures,Precision_at_5,Processing Time,Recall_at_5,Retrieval Score,Samples Per Second,Samples Processed,Throughput,Total
1,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,256,32,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7378 [0.7378, 0.7378]","0.8667 [0.8667, 0.8667]","0.7939 [0.7939, 0.7939]","1.00 [1.00, 1.00]","0.8681 [0.8681, 0.8681]",0.0000,"0.7944 [0.7944, 0.7944]",616.05 seconds,"0.8500 [0.8500, 0.8500]","0.8128 [0.8128, 0.8128]",0.07,45,0.1/s,45
2,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7778 [0.7778, 0.7778]","0.9111 [0.9111, 0.9111]","0.7600 [0.7600, 0.7600]","0.9556 [0.9556, 0.9556]","0.8815 [0.8815, 0.8815]",0.0000,"0.7630 [0.7630, 0.7630]",558.15 seconds,"0.8056 [0.8056, 0.8056]","0.7762 [0.7762, 0.7762]",0.08,45,0.1/s,45
3,api-llama-4-scout,COMPLETED,4/4,0.000,TextSplitterWithLineNum,256,32,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.6844 [0.6844, 0.6844]","0.8444 [0.8444, 0.8444]","0.7867 [0.7867, 0.7867]","1.00 [1.00, 1.00]","0.8430 [0.8430, 0.8430]",0.0000,"0.7815 [0.7815, 0.7815]",519.05 seconds,"0.8389 [0.8389, 0.8389]","0.8023 [0.8023, 0.8023]",0.09,45,0.1/s,45
4,api-llama-4-scout,COMPLETED,4/4,0.000,TextSplitterWithLineNum,512,64,HuggingFaceEmbeddings,32,True,sentence-transformers/all-mpnet-base-v2,32,faiss,15,similarity,CrossEncoderReranker,cross-encoder/ms-marco-MiniLM-L6-v2,5,"max_completion_tokens=2048, reasoning_effort=high","0.7422 [0.7422, 0.7422]","0.9111 [0.9111, 0.9111]","0.7319 [0.7319, 0.7319]","1.00 [1.00, 1.00]","0.8844 [0.8844, 0.8844]",0.0000,"0.7204 [0.7204, 0.7204]",483.02 seconds,"0.8056 [0.8056, 0.8056]","0.7526 [0.7526, 0.7526]",0.09,45,0.1/s,45


Experiment rag-grid-search-4configs_4 ended


In [26]:
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

pd.set_option('display.max_colwidth', None)

results_df.to_csv('./experiments_0.csv', index=False)

results_df

,run_id,model_name,text_splitter,chunk_size,chunk_overlap,embedding_cfg,vector_store_cfg,search_cfg,reranker_cfg,model_config,Samples Processed,Processing Time,Samples Per Second,Total,F1_at_5,Precision_at_5,Recall_at_5,Retrieval Score,Correctness_pass_rate,Faithfulness_pass_rate,Completeness_normalized,Generation_Score_3_released,Judge Failures
0,1,api-mistral-small-3.2-2506,TextSplitterWithLineNum,256,32,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-mpnet-base-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,616.05 seconds,0.07,45,0.793862,0.794444,0.850000,0.812769,0.866667,1.000000,0.737778,0.868148,0
1,2,api-mistral-small-3.2-2506,TextSplitterWithLineNum,512,64,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-mpnet-base-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,558.15 seconds,0.08,45,0.760000,0.762963,0.805556,0.776173,0.911111,0.955556,0.777778,0.881481,0
2,3,api-llama-4-scout,TextSplitterWithLineNum,256,32,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-mpnet-base-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,519.05 seconds,0.09,45,0.786667,0.781481,0.838889,0.802346,0.844444,1.000000,0.684444,0.842963,0
3,4,api-llama-4-scout,TextSplitterWithLineNum,512,64,"{'class': 'HuggingFaceEmbeddings', 'model_name': 'sentence-transformers/all-mpnet-base-v2', 'model_kwargs': {'device': 'cpu'}, 'encode_kwargs': {'normalize_embeddings': True, 'batch_size': 32}}","{'type': 'faiss', 'batch_size': 32}","{'type': 'similarity', 'k': 15}","{'class': 'CrossEncoderReranker', 'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'model_kwargs': {'device': 'cpu'}, 'top_n': 5}","{'max_completion_tokens': 2048, 'reasoning_effort': 'high'}",45,483.02 seconds,0.09,45,0.731852,0.720370,0.805556,0.752593,0.911111,1.000000,0.742222,0.884444,0
